# Topology Final Evaluation

Ordered final evaluation: consolidate completed runs; match the recorded environment; reproduce Calibration; evaluate frozen Test checkpoints; export tables. Requires the existing DDI_V2 data and run directory. No training is performed. Restart after any dependency installation, as instructed by the environment cell.


## Stage 1

Source: `topology_history`, cell 31 (zero-based).


In [ ]:
# CELL 1 — Mount Drive, collect the 9 completed runs, and retain Kaggle backups.
from pathlib import Path
import json, hashlib, os, shutil, zipfile
from google.colab import drive, files

if not Path('/content/drive/MyDrive/DDI_V2').exists():
    drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/DDI_V2')
FEATURES = BASE / 'PRIMARY_TOPOLOGY_TRAIN_ONLY_V2'
FINAL = BASE / 'TOPOLOGY_FINAL_COMMENT7'
RUNS = FINAL / 'runs'
BACKUPS = FINAL / 'kaggle_backups'
for directory in [FINAL, RUNS, BACKUPS]:
    directory.mkdir(parents=True, exist_ok=True)
CONDITIONS = ['original', 'rewired', 'self_loops_only']
SEEDS = [42, 43, 44]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def save_json(path, obj):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(obj, indent=2, allow_nan=False))
    os.replace(temporary, path)


# Copy completed Drive runs into a separate consolidated folder.
# Existing training folders are never edited.
for condition in CONDITIONS:
    for seed in SEEDS:
        source = FEATURES / 'training_current_node_bn_v2' / condition / f'seed_{seed}'
        destination = RUNS / condition / f'seed_{seed}'
        if (source / 'TRAINING_COMPLETE.json').is_file():
            if destination.exists():
                for name in ['config.json', 'TRAINING_COMPLETE.json', 'best_by_macro_f1.pt']:
                    assert (destination / name).is_file()
                    assert sha(source / name) == sha(destination / name), (
                        f'Conflicting consolidated run: {condition} {seed} {name}'
                    )
            else:
                shutil.copytree(source, destination, ignore=shutil.ignore_patterns('*.tmp'))

archive_names = [
    'KAGGLE_SEED43_REWIRED_SELF_LOOPS_COMPLETE.zip',
    'KAGGLE_SEED44_ALL3_COMPLETE.zip'
]
# A fully assembled run set makes uploading the ZIPs again unnecessary.
all_present = all((RUNS / c / f'seed_{s}' / 'TRAINING_COMPLETE.json').is_file()
                  for c in CONDITIONS for s in SEEDS)
if not all_present:
    missing_archives = []
    for name in archive_names:
        destination = BACKUPS / name
        if not destination.exists():
            candidates = sorted(BASE.rglob(name))
            if not candidates:
                root_candidate = Path('/content/drive/MyDrive') / name
                if root_candidate.is_file():
                    candidates = [root_candidate]
            if candidates:
                assert len({sha(p) for p in candidates}) == 1, f'Different backups named {name}'
                shutil.copy2(candidates[0], destination)
            else:
                missing_archives.append(name)
    if missing_archives:
        print('Upload these downloaded Kaggle backups together:', missing_archives)
        uploaded = files.upload()
        for name in missing_archives:
            assert name in uploaded, f'Missing upload: {name}'
            (BACKUPS / name).write_bytes(uploaded[name])
        del uploaded

    for name in archive_names:
        archive_path = BACKUPS / name
        with zipfile.ZipFile(archive_path) as archive:
            assert archive.testzip() is None, f'Damaged archive: {name}'
            prefix = 'DDI_V2/PRIMARY_TOPOLOGY_TRAIN_ONLY_V2/training_current_node_bn_v2/'
            allowed = {'config.json', 'TRAINING_COMPLETE.json', 'last_training_state.pt',
                       'best_by_macro_f1.pt', 'best_by_accuracy.pt', 'best_by_weighted_f1.pt',
                       'training_history.csv', 'execution_segments.json'}
            for member in archive.infolist():
                if member.is_dir() or not member.filename.startswith(prefix):
                    continue
                relative = Path(member.filename[len(prefix):])
                if len(relative.parts) != 3:
                    continue
                condition, seed_folder, filename = relative.parts
                if condition not in CONDITIONS or seed_folder not in ['seed_42','seed_43','seed_44']:
                    continue
                if filename not in allowed:
                    continue
                destination = RUNS / relative
                contents = archive.read(member)
                destination.parent.mkdir(parents=True, exist_ok=True)
                if destination.exists():
                    assert sha(destination) == hashlib.sha256(contents).hexdigest(), (
                        f'Conflicting existing artifact: {relative}'
                    )
                else:
                    temporary = destination.with_name(destination.name + '.tmp')
                    temporary.write_bytes(contents)
                    os.replace(temporary, destination)

records = []
for condition in CONDITIONS:
    for seed in SEEDS:
        run = RUNS / condition / f'seed_{seed}'
        for name in ['config.json', 'TRAINING_COMPLETE.json', 'best_by_macro_f1.pt', 'training_history.csv']:
            assert (run / name).is_file(), f'Missing: {run / name}'
        config = json.loads((run / 'config.json').read_text())
        result = json.loads((run / 'TRAINING_COMPLETE.json').read_text())
        assert config['condition'] == result['condition'] == condition
        assert config['seed'] == result['seed'] == seed
        assert result['protocol'] == config['protocol'] == 'current_node_bn_v2'
        assert result['completed_epochs'] == 120
        assert result['primary_checkpoint'] == 'best_by_macro_f1.pt'
        assert result['config_sha256'] == sha(run / 'config.json')
        assert result['test_evaluated'] is False
        graph_name = ('original.npz' if condition == 'original' else
                      f'rewired_seed_{seed}.npz' if condition == 'rewired' else 'self_loops_only.npz')
        graph = FEATURES / 'topology_graphs_v1' / graph_name
        assert graph.is_file(), f'Missing graph: {graph}'
        assert sha(graph) == config['graph_sha256']
        records.append({
            'condition': condition, 'seed': seed,
            'run': str(run), 'graph': str(graph),
            'checkpoint': str(run / 'best_by_macro_f1.pt'),
            'checkpoint_sha256': sha(run / 'best_by_macro_f1.pt'),
            'config_sha256': sha(run / 'config.json'),
            'graph_sha256': sha(graph),
            'selected_epoch': result['best_epochs']['macro_f1']
        })
selection_path = FINAL / 'SELECTED_CHECKPOINTS.json'
if selection_path.exists():
    assert json.loads(selection_path.read_text()) == records, 'Previously selected checkpoints changed.'
else:
    save_json(selection_path, records)
print('All 9 completed runs collected and checkpoint identities frozen ')
print('Saved folder:', FINAL)
print('No Test data read; no training performed.')


## Stage 2

Source: `topology_history`, cell 32 (zero-based).


In [ ]:
# CELL 2 — Match package versions; GPU optional for inference.
from pathlib import Path
import json, sys, subprocess
import importlib.metadata as md
BASE = Path('/content/drive/MyDrive/DDI_V2')
FINAL = BASE / 'TOPOLOGY_FINAL_COMMENT7'
reference = json.loads((FINAL / 'runs/original/seed_42/config.json').read_text())
expected = reference['versions']
packages = ['torch','torch-geometric','numpy','pandas','scikit-learn']
def installed(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None
changes = {p:[installed(p),expected[p]] for p in packages if installed(p)!=expected[p]}
print('Required environment changes:', changes)
if changes:
    if 'torch' in changes:
        assert expected['torch'].endswith('+cu128')
        subprocess.check_call([sys.executable,'-m','pip','install','--upgrade',
                               'torch=='+expected['torch'],'--index-url','https://download.pytorch.org/whl/cu128'])
    subprocess.check_call([sys.executable,'-m','pip','install','--upgrade',
                          *[p+'=='+expected[p] for p in packages if p!='torch']])
    for p in packages:
        assert installed(p)==expected[p], f'Package mismatch: {p}'
    raise RuntimeError('Installation complete. Restart session once, then run from cell 1 again.')
import torch
print('Environment matched ')
print('Inference device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## Stage 3

Source: `topology_history`, cell 33 (zero-based).


In [ ]:
# CELL 3 — Validate all configurations, histories, graphs, and Calibration metrics.
from pathlib import Path
import json, hashlib, os, gc
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from tqdm import tqdm
from datetime import datetime, timezone

BASE = Path('/content/drive/MyDrive/DDI_V2')
FEATURES = BASE / 'PRIMARY_TOPOLOGY_TRAIN_ONLY_V2'
FINAL = BASE / 'TOPOLOGY_FINAL_COMMENT7'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CLASSES = 84
CAL_TOLERANCE = 1e-6  # metric fractions, fixed before inspecting Test

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

def sha(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def save_json(path, obj):
    temporary = path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(obj,indent=2,allow_nan=False))
    os.replace(temporary,path)

def save_csv(path, frame):
    temporary = path.with_name(path.name+'.tmp')
    frame.to_csv(temporary,index=False)
    os.replace(temporary,path)

records = json.loads((FINAL/'SELECTED_CHECKPOINTS.json').read_text())
assert len(records)==9
reference = json.loads((FINAL/'runs/original/seed_42/config.json').read_text())
import importlib.metadata as md
for package, version in reference['versions'].items():
    assert md.version(package)==version, f'Run environment setup first: {package}'
feature_path = FEATURES/'train_only_feature_cache.npz'
assert sha(feature_path)==reference['feature_sha256']
with np.load(feature_path,allow_pickle=False) as loaded:
    x = torch.tensor(loaded['x'],dtype=torch.float32,device=DEVICE)
    drugs = loaded['all_drugs'].tolist()
assert x.shape==(1705,2250) and torch.isfinite(x).all()
mapping = {d:i for i,d in enumerate(drugs)}
assert len(mapping)==len(drugs)

# No Test partition is opened in this cell.
def read_pairs(split):
    path = BASE/f'clean_{split}_split.csv'
    frame = pd.read_csv(path)
    a,b=frame['Drug1'].map(mapping),frame['Drug2'].map(mapping)
    assert a.notna().all() and b.notna().all(), f'Unknown drug in {split}'
    y=frame['label_id'].to_numpy(dtype=np.int64)
    assert len(frame)==(134249 if split=='train' else 28768)
    assert set(y)==set(range(N_CLASSES))
    pairs=torch.tensor(np.column_stack([a,b]).astype(np.int64),dtype=torch.long,device=DEVICE)
    return pairs,y

for split in ['train','calibration']:
    assert sha(BASE/f'clean_{split}_split.csv')==reference['split_sha256'][split]
train_pairs,train_truth=read_pairs('train')
cal_pairs,cal_truth=read_pairs('calibration')
counts=np.bincount(train_truth,minlength=84).astype(float)
weights=np.clip(np.sqrt(counts.max()/(counts+1)),0.5,3.0)
weights/=weights.mean()
np.testing.assert_allclose(weights,reference['class_weights'],rtol=0,atol=1e-12)

class CurrentNodeBNGCN(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [2250, 384, 384, 192]
        self.convs = nn.ModuleList([
            GCNConv(
                dims[i],
                dims[i + 1],
                improved=False,
                cached=False,
                add_self_loops=False,
                normalize=True,
                bias=True
            )
            for i in range(3)
        ])
        self.norms = nn.ModuleList([
            nn.BatchNorm1d(d, track_running_stats=False)
            for d in [384, 384, 192]
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 84)
        )
    def encode(self, x, edges):
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            x = norm(conv(x, edges))
            if i < 2:
                x = F.dropout(
                    F.relu(x),
                    p=0.35,
                    training=self.training
                )
        return x
    def classify(self, z, pairs):
        a = z[pairs[:, 0]]
        b = z[pairs[:, 1]]
        pair_features = torch.cat(
            [a, b, (a - b).abs(), a * b],
            dim=1
        )
        return self.head(pair_features)
    def forward(self, x, edges, pairs):
        return self.classify(self.encode(x, edges), pairs)


@torch.inference_mode()
def predict(model, edges, pairs):
    model.eval()
    assert all(not m.track_running_stats for m in model.modules() if isinstance(m,nn.BatchNorm1d))
    z=model.encode(x,edges)
    assert torch.isfinite(z).all()
    predictions=[]
    for start in range(0,len(pairs),2048):
        logits=model.classify(z,pairs[start:start+2048])
        assert torch.isfinite(logits).all()
        predictions.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(predictions)

def metrics(truth,pred):
    return {
        'accuracy':float(accuracy_score(truth,pred)),
        'macro_f1':float(f1_score(truth,pred,labels=np.arange(84),average='macro',zero_division=0)),
        'weighted_f1':float(f1_score(truth,pred,labels=np.arange(84),average='weighted',zero_division=0))
    }

def load_job(record):
    config_path=Path(record['run'])/'config.json'
    checkpoint_path=Path(record['checkpoint'])
    graph_path=Path(record['graph'])
    assert sha(config_path)==record['config_sha256']
    assert sha(checkpoint_path)==record['checkpoint_sha256']
    assert sha(graph_path)==record['graph_sha256']
    config=json.loads(config_path.read_text())
    with np.load(graph_path,allow_pickle=False) as loaded:
        edges=torch.tensor(loaded['edge_index'],dtype=torch.long,device=DEVICE)
    assert edges.ndim==2 and edges.shape[0]==2 and edges.numel()>0
    assert int(edges.min())>=0 and int(edges.max())<len(drugs)
    if record['condition']=='self_loops_only':
        assert edges.shape[1]==len(drugs) and torch.equal(edges[0],edges[1])
        assert torch.equal(torch.sort(edges[0]).values,torch.arange(len(drugs),device=DEVICE))
    saved=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    assert saved['config_sha256']==record['config_sha256']
    assert saved['criterion']=='macro_f1'
    assert saved['epoch']==record['selected_epoch']
    assert saved['test_evaluated'] is False
    model=CurrentNodeBNGCN().to(DEVICE)
    assert str(model)==config['architecture']
    model.load_state_dict(saved['model'],strict=True)
    return model,edges,saved

common_keys=['protocol','epochs','batch_size','optimizer','learning_rate','weight_decay',
             'gradient_clip_norm','early_stopping','scheduler','architecture','class_weights',
             'primary_selection','tie_rule','BN','BN_track_running_stats','BN_node_scope',
             'descriptor_preprocessing_fit','shuffle_seed','feature_sha256','split_sha256','versions']
rows=[]
for record in tqdm(records,desc='Calibration reproduction 9/9'):
    run=Path(record['run'])
    config=json.loads((run/'config.json').read_text())
    result=json.loads((run/'TRAINING_COMPLETE.json').read_text())
    for key in common_keys:
        assert config[key]==reference[key], f'Configuration mismatch: {record["condition"]} {record["seed"]} {key}'
    assert config['seed']==record['seed'] and config['condition']==record['condition']
    assert result['completed_epochs']==config['epochs']==120
    history=pd.read_csv(run/'training_history.csv')
    assert history['epoch'].tolist()==list(range(1,121))
    best_row=history.loc[history['cal_macro_f1'].idxmax()]
    assert int(best_row['epoch'])==record['selected_epoch'], 'Primary epoch does not match earliest Calibration maximum.'
    model,edges,saved=load_job(record)
    pred=predict(model,edges,cal_pairs)
    measured=metrics(cal_truth,pred)
    row={'condition':record['condition'],'seed':record['seed'],'selected_epoch':saved['epoch']}
    for metric in ['accuracy','macro_f1','weighted_f1']:
        expected_value=saved['calibration_metrics'][metric]
        difference=measured[metric]-expected_value
        row['recorded_'+metric]=expected_value
        row['recomputed_'+metric]=measured[metric]
        row['difference_'+metric]=difference
    assert abs(saved['calibration_metrics']['macro_f1']-result['best_calibration_scores']['macro_f1'])<=1e-12
    assert abs(saved['calibration_metrics']['macro_f1']-float(best_row['cal_macro_f1']))<=1e-12
    row['passed']=all(abs(row['difference_'+m])<=CAL_TOLERANCE for m in ['accuracy','macro_f1','weighted_f1'])
    rows.append(row)
    save_csv(FINAL/'CALIBRATION_REPRODUCTION.csv',pd.DataFrame(rows))
    np.save(FINAL/f'calibration_predictions_{record["condition"]}_{record["seed"]}.npy',pred)
    del model,edges,saved,pred
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
report=pd.DataFrame(rows)
display(report)
assert report['passed'].all(), 'Calibration reproduction failed. Stop before Test and inspect CALIBRATION_REPRODUCTION.csv.'

# Graph audit, including whether Original equals the undirected Train-pair graph plus self-loops.
original_graph=FEATURES/'topology_graphs_v1/original.npz'
with np.load(original_graph,allow_pickle=False) as loaded:
    original_edges=loaded['edge_index']
original_degree=np.bincount(original_edges[0],minlength=len(drugs))
train_np=train_pairs.cpu().numpy()
train_graph_keys=np.unique(np.concatenate([
    train_np[:,0]*len(drugs)+train_np[:,1],
    train_np[:,1]*len(drugs)+train_np[:,0],
    np.arange(len(drugs))*(len(drugs)+1)
]))
audit=[]
for graph_path in sorted({r['graph'] for r in records}):
    with np.load(graph_path,allow_pickle=False) as loaded:
        e=loaded['edge_index']
    keys=e[0]*len(drugs)+e[1]
    audit.append({
        'graph':Path(graph_path).name,'sha256':sha(graph_path),
        'directed_edge_entries':int(e.shape[1]),'unique_directed_edges':int(len(np.unique(keys))),
        'self_loop_entries':int(np.sum(e[0]==e[1])),
        'out_degree_matches_original':bool(np.array_equal(np.bincount(e[0],minlength=len(drugs)),original_degree)),
        'equals_undirected_train_pairs_plus_self_loops':bool(np.array_equal(np.unique(keys),train_graph_keys))
    })
save_csv(FINAL/'GRAPH_AUDIT.csv',pd.DataFrame(audit))
pass_record={
    'selection_sha256':sha(FINAL/'SELECTED_CHECKPOINTS.json'),
    'calibration_sha256':sha(BASE/'clean_calibration_split.csv'),
    'feature_sha256':sha(feature_path),'calibration_report_sha256':sha(FINAL/'CALIBRATION_REPRODUCTION.csv'),
    'all_9_passed':True,'tolerance_fraction':CAL_TOLERANCE,
    'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
    'utc':datetime.now(timezone.utc).isoformat(),'test_evaluated':False
}
save_json(FINAL/'CALIBRATION_AUDIT_PASS.json',pass_record)
print('All 9 checkpoints reproduced Calibration; selection is frozen ')
print('Test has not been opened for metric evaluation.')


## Stage 4

Source: `topology_history`, cell 34 (zero-based).


In [ ]:
# CELL 4 — Evaluate the frozen 9 checkpoints on the existing Test partition.
assert 'records' in globals() and 'load_job' in globals(), 'Run cell 3 first.'
passed=json.loads((FINAL/'CALIBRATION_AUDIT_PASS.json').read_text())
assert passed['all_9_passed'] is True
assert passed['selection_sha256']==sha(FINAL/'SELECTED_CHECKPOINTS.json')
assert passed['calibration_report_sha256']==sha(FINAL/'CALIBRATION_REPRODUCTION.csv')
assert passed['feature_sha256']==sha(feature_path)
for record in records:
    assert sha(Path(record['checkpoint']))==record['checkpoint_sha256']
    assert sha(Path(record['run'])/'config.json')==record['config_sha256']
    assert sha(Path(record['graph']))==record['graph_sha256']

test_path=BASE/'clean_test_split.csv'
assert test_path.is_file(), f'Missing fixed Test split: {test_path}'
test_hash=sha(test_path)
if 'test' in reference['split_sha256']:
    assert test_hash==reference['split_sha256']['test']
freeze={
    'selection_sha256':sha(FINAL/'SELECTED_CHECKPOINTS.json'),
    'test_sha256':test_hash,'feature_sha256':sha(feature_path),
    'protocol':'current_node_bn_v2','primary_checkpoint':'best_by_macro_f1.pt',
    'macro_label_space':list(range(84)),
    'selection':'Calibration macro-F1 only, earliest maximum over 120 fixed epochs',
    'test_based_tuning':False,'node_scope':'1705 fixed nodes; transductive current-node BN',
    'descriptor_fit':'Train nodes only',
    'historical_test_exposure':'The manuscript discloses prior Test use in a separate historical meta-logistic search; this is not an untouched external validation set.'
}
freeze_path=FINAL/'FROZEN_TEST_PROTOCOL.json'
if freeze_path.exists():
    assert json.loads(freeze_path.read_text())==freeze, 'Frozen evaluation protocol changed.'
else:
    save_json(freeze_path,freeze)
test_pairs,test_truth=read_pairs('test')
RESULTS=FINAL/'test_results'
RESULTS.mkdir(exist_ok=True)
rows=[]
for record in tqdm(records,desc='Frozen Test evaluation 9/9'):
    tag=f'{record["condition"]}_seed_{record["seed"]}'
    identity={'checkpoint_sha256':record['checkpoint_sha256'],
              'graph_sha256':record['graph_sha256'],'config_sha256':record['config_sha256'],
              'test_sha256':test_hash,'protocol_sha256':sha(freeze_path)}
    result_file=RESULTS/f'{tag}_metrics.json'
    pred_file=RESULTS/f'{tag}_predictions.npy'
    if result_file.exists():
        stored=json.loads(result_file.read_text())
        assert stored['identity']==identity
        assert pred_file.is_file() and sha(pred_file)==stored['predictions_sha256']
        pred=np.load(pred_file,allow_pickle=False)
        assert pred.shape==test_truth.shape
        measured=metrics(test_truth,pred)
        assert measured==stored['metrics']
        print('Reused saved Test result:',tag)
    else:
        model,edges,saved=load_job(record)
        pred=predict(model,edges,test_pairs)
        measured=metrics(test_truth,pred)
        np.save(pred_file,pred)
        save_json(result_file,{
            'identity':identity,'condition':record['condition'],'seed':record['seed'],
            'selected_epoch':record['selected_epoch'],'metrics':measured,
            'predictions_sha256':sha(pred_file),'test_evaluated':True,
            'device':str(DEVICE),'utc':datetime.now(timezone.utc).isoformat()
        })
        del model,edges,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    precision,recall,f1,support=precision_recall_fscore_support(
        test_truth,pred,labels=np.arange(84),zero_division=0)
    save_csv(RESULTS/f'{tag}_per_class.csv',pd.DataFrame({
        'class_id':np.arange(84),'precision':precision,'recall':recall,'f1':f1,'support':support
    }))
    np.save(RESULTS/f'{tag}_confusion_matrix.npy',confusion_matrix(test_truth,pred,labels=np.arange(84)))
    rows.append({'condition':record['condition'],'seed':record['seed'],
                 'selected_epoch':record['selected_epoch'],'rows':len(test_truth),**measured})
    save_csv(FINAL/'TEST_RESULTS_9_RUNS.csv',pd.DataFrame(rows))
report=pd.DataFrame(rows)
assert len(report)==9
display(report)
summary=report.groupby('condition',sort=False)[['accuracy','macro_f1','weighted_f1']].agg(['mean','std'])
summary.columns=['_'.join(c) for c in summary.columns]
summary=summary.reset_index()
save_csv(FINAL/'TEST_SUMMARY_MEAN_SAMPLE_SD.csv',summary)
display(summary)
paired=report.pivot(index='seed',columns='condition',values='macro_f1')
paired['self_minus_original']=paired['self_loops_only']-paired['original']
paired['original_minus_rewired']=paired['original']-paired['rewired']
save_csv(FINAL/'PAIRED_MACRO_F1_DIFFERENCES.csv',paired.reset_index())
save_json(FINAL/'TEST_EVALUATION_COMPLETE.json',{
    'runs':9,'protocol_sha256':sha(freeze_path),'test_sha256':test_hash,
    'test_results_sha256':sha(FINAL/'TEST_RESULTS_9_RUNS.csv'),
    'utc':datetime.now(timezone.utc).isoformat(),
    'training_performed':False,'test_based_tuning':False
})
print('All 9 Test evaluations saved. No retraining or Test-based tuning ')


## Stage 5

Source: `topology_history`, cell 35 (zero-based).


In [ ]:
# CELL 5 — Export the evidence bundle and print the manuscript table.
from pathlib import Path
import json, os, zipfile
from google.colab import files
assert (FINAL/'TEST_EVALUATION_COMPLETE.json').is_file(), 'Complete Test evaluation first.'
summary=pd.read_csv(FINAL/'TEST_SUMMARY_MEAN_SAMPLE_SD.csv').set_index('condition')
print('\n=== FINAL TEST TABLE — mean ± sample SD, percent ===')
for condition in ['original','rewired','self_loops_only']:
    row=summary.loc[condition]
    values=[f"{100*row[m+'_mean']:.2f} ± {100*row[m+'_std']:.2f}" for m in ['accuracy','macro_f1','weighted_f1']]
    print(condition,'| Accuracy:',values[0],'| Macro-F1:',values[1],'| Weighted-F1:',values[2])
notes='Topology ablation evidence for Comment 7\nNine matched runs use seeds 42, 43, 44, a fixed pair-level split, 120 epochs, Train-fitted descriptor preprocessing, identical class weights and four-way pair features, and current full-node BatchNorm in training and inference. Node features include the same 1705 fixed nodes, so the normalization scope is transductive. Dropout is disabled during inference. Each primary checkpoint was selected only by Calibration macro-F1 over all 84 classes, with the earliest epoch used for ties.\nThe fixed Test partition has prior historical exposure disclosed in the manuscript; this analysis is not untouched external validation. Results describe topology sensitivity under this protocol, not all graph neural networks or clinical utility. Sample SD summarizes training-seed variability on one split; no significance claim follows from three seeds.\nThe previous saved-running-stat BatchNorm runs remain separate preliminary diagnostics. Their results must not be mixed with current_node_bn_v2.\nReview GRAPH_AUDIT.csv before claiming degree preservation or exact Train-edge provenance. Test inference does not modify checkpoint files. The transfer of Rewired 43 between Colab and Kaggle is logged in its execution_segments.json and is not guaranteed bitwise identical to uninterrupted training.\n'
(FINAL/'REPORTING_NOTES.txt').write_text(notes)
# Generate factual English insertion text from the actual Test results.
def format_metric(condition, metric):
    row = summary.loc[condition]
    return f"{100*row[metric+'_mean']:.2f} ± {100*row[metric+'_std']:.2f}%"
method_text = (
    "A matched graph-topology sensitivity analysis compared the original Train-derived graph, "
    "a rewired graph, and self-loops only across training seeds 42, 43, and 44 on the same fixed split. "
    "The 2250-dimensional node representation, class weighting, three-layer encoder, four-way pair "
    "operator, AdamW settings, batch size, and fixed 120-epoch budget were held constant. "
    "Batch normalization used current statistics of the same 1705-node matrix in training and inference "
    "(track_running_stats=False); dropout was disabled in inference. Descriptor preprocessing was "
    "fitted on Train nodes, whereas the normalization scope was transductive. Each primary checkpoint "
    "was the earliest epoch maximizing Calibration macro-F1 over all 84 classes. All nine checkpoints "
    "passed Calibration reproduction before the fixed Test evaluation; no Test-based retuning was used. "
    "The historical Test exposure disclosed elsewhere in the manuscript remains applicable."
)
results_text = (
    "Across the three training seeds, the original graph achieved Test accuracy, macro-F1, and "
    "weighted-F1 of " + ', '.join(format_metric('original',m) for m in ['accuracy','macro_f1','weighted_f1']) +
    ", respectively. The rewired graph achieved " +
    ', '.join(format_metric('rewired',m) for m in ['accuracy','macro_f1','weighted_f1']) +
    ", and self-loops only achieved " +
    ', '.join(format_metric('self_loops_only',m) for m in ['accuracy','macro_f1','weighted_f1']) +
    ", respectively. Values are mean ± sample SD across training seeds on one fixed split. "
    "These comparisons isolate topology sensitivity within this corrected protocol; they do not "
    "constitute a complete factorial ablation or establish general clinical utility."
)
(FINAL/'COMMENT7_METHODS_INSERT.txt').write_text(method_text)
(FINAL/'COMMENT7_RESULTS_INSERT.txt').write_text(results_text)
(FINAL/'COMMENT7_RESPONSE_EVIDENCE.txt').write_text(
    "We performed a matched original/rewired/self-loops-only topology analysis across seeds 42, 43, "
    "and 44, holding the representation, class weighting, pair operator, and training budget fixed. "
    "Checkpoints were selected using Calibration macro-F1 and verified before Test evaluation. "
    + results_text + " " +
    "The experiments use the corrected current-node BatchNorm protocol and are reported separately "
    "from historical-cache and saved-running-stat experiments. We retain the limitations of "
    "transductive normalization, historical Test exposure, and the absence of a full factorial study."
)
# Include complete consolidated runs and reports; exclude duplicate imported archives.
output=BASE/'TOPOLOGY_COMMENT7_FINAL_EVIDENCE.zip'
local_output=Path('/content/TOPOLOGY_COMMENT7_FINAL_EVIDENCE.zip')
temporary=local_output.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as archive:
    for path in sorted(FINAL.rglob('*')):
        if not path.is_file() or path.name.endswith('.tmp') or BACKUPS in path.parents:
            continue
        archive.write(path,arcname=(Path('TOPOLOGY_FINAL_COMMENT7')/path.relative_to(FINAL)).as_posix())
    for name in ['graph_protocol.json','graph_audit.csv']:
        path=FEATURES/'topology_graphs_v1'/name
        if path.is_file():
            archive.write(path,arcname='TOPOLOGY_FINAL_COMMENT7/graph_provenance/'+name)
with zipfile.ZipFile(temporary) as archive:
    assert archive.testzip() is None
os.replace(temporary,local_output)
import shutil
shutil.copy2(local_output,output)
print('\nEvidence bundle saved to Drive:',output)
print('Size MB:',round(local_output.stat().st_size/1024**2,2))
print('Send TEST_RESULTS_9_RUNS.csv, TEST_SUMMARY_MEAN_SAMPLE_SD.csv and GRAPH_AUDIT.csv for the final manuscript/reviewer wording.')
files.download(str(local_output))
